In [1]:
import pandas as pd

attrition = pd.read_csv('attrition_log.csv')
employees = pd.read_csv('employees.csv')
engagement = pd.read_csv('engagement.csv')
performance = pd.read_csv('performance.csv')

print("Employees:", employees.shape)
print("Attrition:", attrition.shape)
print("Engagement:", engagement.shape)
print("Performance:", performance.shape)

Employees: (13403, 24)
Attrition: (1400, 10)
Engagement: (55971, 12)
Performance: (34979, 7)


In [2]:
def run_eda(df, name):
    summary = pd.DataFrame({
        "dtype": df.dtypes,
        "missing": df.isna().sum(),
        "missing_%":df.isna().mean() *100,
        "unique": df.nunique()
    }
    )
    print(f"{name}")
    print(summary)

In [3]:
run_eda(employees, "employees")

employees
                       dtype  missing  missing_%  unique
employee_id           object        0   0.000000   13403
name                  object        0   0.000000   12378
hire_date             object        0   0.000000    6588
exit_date             object    12003  89.554577     480
status                object        0   0.000000       2
department            object        0   0.000000       7
role_family           object        0   0.000000       7
role_level             int64        0   0.000000       8
job_title             object        0   0.000000      65
salary               float64        0   0.000000    2065
compa_ratio          float64        0   0.000000      53
gender                object        0   0.000000       4
age_band              object        0   0.000000       9
cultural_background   object        0   0.000000      11
contract_type         object        0   0.000000       4
hipo_flag               bool        0   0.000000       2
promotion_eligible   

In [4]:
run_eda(performance, "performance")

performance
                            dtype  missing  missing_%  unique
employee_id                object        0        0.0   13294
review_date                object        0        0.0     277
performance_rating         object        0        0.0       5
review_cycle               object        0        0.0       3
promotion_recommendation     bool        0        0.0       2
goal_achievement_score    float64        0        0.0     893
reviewer_id                object        0        0.0    4899


In [ ]:
LEAVER_STATUS_VALUES = employees["status"].unique().tolist()
print(LEAVER_STATUS_VALUES)

number_who_left = employees["status"].str.lower()=="departed"
print("The people who left",number_who_left.sum())

stayed= employees["status"].str.lower()=="active"
print("People who stayed",stayed.sum())

employees["is_leaver"] = employees["status"].str.lower() == "departed"

['active', 'departed']
The people who left 1400
People who stayed 12003


In [5]:
print(employees["hire_date"].head(10))

0    1988-01-11
1    1988-01-13
2    1988-01-14
3    1988-01-14
4    1988-01-15
5    1988-01-15
6    1988-01-18
7    1988-01-20
8    1988-01-23
9    1988-01-23
Name: hire_date, dtype: object


In [7]:
print("PERFORMANCE PROFILE: LEAVERS VS STAYERS")

# Check the actual rating labels before mapping
print("performance_rating values:", performance["performance_rating"].unique())

PERFORMANCE PROFILE: LEAVERS VS STAYERS
performance_rating values: ['Meets Expectations' 'High Performer' 'Unsatisfactory' 'Outstanding'
 'Below Expectations']


In [8]:
RATING_ORDER = {
    "Unsatisfactory": 1,
    "Below Expectations": 2,
    "Meets Expectations": 3,
    "High Performer": 4,
    "Outstanding": 5,
}
performance["review_date"] = pd.to_datetime(performance['review_date'])
performance["rating_score"] = performance["performance_rating"].map(RATING_ORDER)

#check for NaN values
assert performance["rating_score"].isna().sum() == 0, "Unmapped rating label found"

# Merge performance with leaver status + exit info
performace_leavers = performance.merge(employees[["employee_id", "is_leaver", "department"]], on="employee_id", how="left")

KeyError: "['is_leaver'] not in index"

In [ ]:
from scipy import stats
print("2. EXIT PROFILE")
 
print("-- Voluntary vs Involuntary --")
print(attrition["exit_type"].value_counts(normalize=True).mul(100).round(1))
 
print("\n-- Regrettable vs Non-regrettable --")
print(attrition["regrettable_flag"].value_counts(normalize=True).mul(100).round(1))
 
print("\n-- Top stated exit reasons --")
print(attrition["stated_exit_reason"].value_counts().head(10))
 
# Is exit_type (voluntary/involuntary) associated with department?
attr_dept = attrition.merge(
    employees[["employee_id", "department", "role_family"]],
    on="employee_id", how="left"
)
ct_dept = pd.crosstab(attr_dept["department"], attr_dept["exit_type"])
chi2, p_dept, dof, expected = stats.chi2_contingency(ct_dept)
print(f"\nChi-square (exit_type x department): chi2={chi2:.2f}, p={p_dept:.4f}")
if p_dept < 0.05:
    print("  -> Statistically significant: exit type distribution differs by department.")
else:
    print("  -> Not statistically significant at alpha=0.05.")

2. EXIT PROFILE
-- Voluntary vs Involuntary --
exit_type
voluntary      80.9
involuntary    19.1
Name: proportion, dtype: float64

-- Regrettable vs Non-regrettable --
regrettable_flag
False    89.1
True     10.9
Name: proportion, dtype: float64

-- Top stated exit reasons --
stated_exit_reason
Career advancement                   649
Better opportunity                   271
Involuntary - performance            196
Compensation                          50
Involuntary - restructure             50
Work-life balance                     45
Personal reasons                      36
Relocation                            33
Role uncertainty / unclear future     30
Involuntary - conduct                 21
Name: count, dtype: int64

Chi-square (exit_type x department): chi2=6.02, p=0.4207
  -> Not statistically significant at alpha=0.05.


In [ ]:
print("3. TIME PATTERNS")
import numpy as np
attrition["exit_date"] = pd.to_datetime(attrition["exit_date"])

monthly_exits = (
    attrition.set_index("exit_date")
    .resample("ME")["employee_id"]
    .count()
    .rename("exits")
)
print("\nMonthly exit counts (first 12):")
print(monthly_exits.head(12))
 
# Simple trend test: is attrition trending up/down over time? (Mann-Kendall via correlation)
t = np.arange(len(monthly_exits))
trend_corr, trend_p = stats.spearmanr(t, monthly_exits.values)
print(f"\nSpearman trend test: rho={trend_corr:.3f}, p={trend_p:.4f}")

3. TIME PATTERNS

Monthly exit counts (first 12):
exit_date
2024-01-31     2
2024-02-29    45
2024-03-31    57
2024-04-30    47
2024-05-31    52
2024-06-30    64
2024-07-31    69
2024-08-31    88
2024-09-30    72
2024-10-31    74
2024-11-30    60
2024-12-31    50
Freq: ME, Name: exits, dtype: int64

Spearman trend test: rho=0.071, p=0.7418


In [ ]:
print(attrition["exit_date"].head())

0   2024-12-24
1   2025-04-17
2   2024-02-19
3   2024-05-24
4   2025-07-04
Name: exit_date, dtype: datetime64[ns]


In [ ]:
print("4. NUMERICAL DISTRIBUTIONS: SALARY")
 
print("\nSalary summary, leavers vs stayers:")
employees["is_leaver"] = employees["status"].str.lower() == "departed"
print(employees.groupby("is_leaver")["salary"].describe()[["count", "mean", "std", "50%"]])
 
print("\nCompa-ratio summary, leavers vs stayers:")
print(employees.groupby("is_leaver")["compa_ratio"].describe()[["count", "mean", "std", "50%"]])
 
# Normality check to decide t-test vs Mann-Whitney
leaver_salary = employees.loc[employees["is_leaver"], "salary"].dropna()
stayer_salary = employees.loc[~employees["is_leaver"], "salary"].dropna()
_, p_norm_leaver = stats.shapiro(leaver_salary.sample(min(500, len(leaver_salary)), random_state=1))
print(f"\nShapiro-Wilk normality (leaver salary sample): p={p_norm_leaver:.4f}"
      f" ({'normal' if p_norm_leaver > 0.05 else 'not normal -> use Mann-Whitney'})")

4. NUMERICAL DISTRIBUTIONS: SALARY

Salary summary, leavers vs stayers:
             count           mean           std       50%
is_leaver                                                
False      12003.0  128353.419978  61213.566188  122700.0
True        1400.0  125558.357143  49366.876045  123100.0

Compa-ratio summary, leavers vs stayers:
             count      mean       std   50%
is_leaver                                   
False      12003.0  0.945764  0.071193  0.95
True        1400.0  0.942257  0.070313  0.94

Shapiro-Wilk normality (leaver salary sample): p=0.0000 (not normal -> use Mann-Whitney)


In [ ]:
print(employees["status"].unique())

['active' 'departed']


In [ ]:
print("5. HYPOTHESIS TESTS")
 
# --- H1: Salary/compa-ratio differs between leavers and stayers -----------
print("\n-- H1: Salary/compa-ratio and attrition --")
u_stat, p_salary = stats.mannwhitneyu(leaver_salary, stayer_salary, alternative="two-sided")
print(f"Mann-Whitney U (salary, leavers vs stayers): U={u_stat:.0f}, p={p_salary:.4f}")
 
leaver_compa = employees.loc[employees["is_leaver"], "compa_ratio"].dropna()
stayer_compa = employees.loc[~employees["is_leaver"], "compa_ratio"].dropna()
u_stat2, p_compa = stats.mannwhitneyu(leaver_compa, stayer_compa, alternative="two-sided")
print(f"Mann-Whitney U (compa_ratio, leavers vs stayers): U={u_stat2:.0f}, p={p_compa:.4f}")
 
# Effect size: rank-biserial correlation (simple, interpretable)
def rank_biserial(u, n1, n2):
    return 1 - (2 * u) / (n1 * n2)
 
r_rb = rank_biserial(u_stat2, len(leaver_compa), len(stayer_compa))
print(f"Effect size (rank-biserial r): {r_rb:.3f}")
 
 

5. HYPOTHESIS TESTS

-- H1: Salary/compa-ratio and attrition --
Mann-Whitney U (salary, leavers vs stayers): U=8362632, p=0.7733
Mann-Whitney U (compa_ratio, leavers vs stayers): U=8185496, p=0.1136
Effect size (rank-biserial r): 0.026


In [ ]:
# --- H2: Job redundancy -- 'pathway' field and department concentration --
print("\n-- H2: Job redundancy (pathway) --")
print("Pathway values:", attrition["pathway"].unique())
print(attrition["pathway"].value_counts(normalize=True).mul(100).round(1))
 
ct_pathway_dept = pd.crosstab(attr_dept["department"], attr_dept["pathway"])
chi2_r, p_redund, dof_r, exp_r = stats.chi2_contingency(ct_pathway_dept)
print(f"\nChi-square (pathway x department): chi2={chi2_r:.2f}, p={p_redund:.4f}")
if p_redund < 0.05:
    top_dept = ct_pathway_dept.idxmax()
    print("  -> Redundancy pathway is NOT evenly spread across departments.")
    print("  -> Department breakdown:\n", ct_pathway_dept)


-- H2: Job redundancy (pathway) --
Pathway values: ['push' 'pull']
pathway
push    68.2
pull    31.8
Name: proportion, dtype: float64

Chi-square (pathway x department): chi2=15.85, p=0.0146
  -> Redundancy pathway is NOT evenly spread across departments.
  -> Department breakdown:
 pathway               pull  push
department                      
Corporate Operations    74   101
Executive Leadership     5    14
Insurance               51   133
Retail Banking         101   205
Risk & Compliance       83   156
Technology              86   229
Wealth Management       45   117


In [ ]:
# --- H3: Working hours (PROXY via engagement wellbeing / role-future confidence) --
print("\n-- H3: Working hours / burnout proxy --")
print("NOTE: no direct 'hours worked' field exists in any dataset.")
print("Using engagement 'wellbeing' and 'confidence_in_role_future' as a proxy.")
print("If a timesheet/HRIS system is available, replace this with actual hours data.")
 
eng_avg = (
    engagement.dropna(subset=["wellbeing", "confidence_in_role_future"])
    .groupby("employee_id")[["wellbeing", "confidence_in_role_future"]]
    .mean()
    .reset_index()
)
eng_merged = eng_avg.merge(employees[["employee_id", "is_leaver"]], on="employee_id", how="left")
 
leaver_wb = eng_merged.loc[eng_merged["is_leaver"] == True, "wellbeing"].dropna()
stayer_wb = eng_merged.loc[eng_merged["is_leaver"] == False, "wellbeing"].dropna()
if len(leaver_wb) > 1 and len(stayer_wb) > 1:
    u_wb, p_wb = stats.mannwhitneyu(leaver_wb, stayer_wb, alternative="two-sided")
    print(f"\nMann-Whitney U (wellbeing, leavers vs stayers): U={u_wb:.0f}, p={p_wb:.4f}")
    print(f"Mean wellbeing -- leavers: {leaver_wb.mean():.2f}, stayers: {stayer_wb.mean():.2f}")


-- H3: Working hours / burnout proxy --
NOTE: no direct 'hours worked' field exists in any dataset.
Using engagement 'wellbeing' and 'confidence_in_role_future' as a proxy.
If a timesheet/HRIS system is available, replace this with actual hours data.

Mann-Whitney U (wellbeing, leavers vs stayers): U=5769064, p=0.1196
Mean wellbeing -- leavers: 3.33, stayers: 3.38


In [ ]:
print("SUMMARY OF TESTS")
summary = pd.DataFrame([
    {"test": "Exit type x Department (chi-square)", "p_value": p_dept},
    {"test": "Attrition trend over time (Spearman)", "p_value": trend_p},
    {"test": "Salary, leavers vs stayers (Mann-Whitney)", "p_value": p_salary},
    {"test": "Compa-ratio, leavers vs stayers (Mann-Whitney)", "p_value": p_compa},
    {"test": "Redundancy pathway x Department (chi-square)", "p_value": p_redund},
])
summary["significant_at_0.05"] = summary["p_value"] < 0.05
print(summary.to_string(index=False))

SUMMARY OF TESTS
                                          test  p_value  significant_at_0.05
           Exit type x Department (chi-square) 0.420708                False
          Attrition trend over time (Spearman) 0.741778                False
     Salary, leavers vs stayers (Mann-Whitney) 0.773290                False
Compa-ratio, leavers vs stayers (Mann-Whitney) 0.113558                False
  Redundancy pathway x Department (chi-square) 0.014576                 True
